In [2]:
## importing the important libraries:
import os
from dotenv import load_dotenv
load_dotenv()

from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence.aio import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature
from azure.ai.documentintelligence.models import DocumentContentFormat
from openai import AsyncAzureOpenAI
import base64


In [3]:
file_path = r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparator\sample\Output 1.pdf"

# Load environment variables
endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")
key = os.getenv("DOCUMENT_INTELLIGENCE_KEY")

In [ ]:
### function for the reading the pdf as input and return the result
document_intelligence_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

async def data_extractor(pdf_path:str):
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()

    # Base64 encode PDF
    base64_encoded_pdf = base64.b64encode(pdf_bytes).decode("utf-8")

    analyze_request = {
        "base64Source": base64_encoded_pdf
    }

    # Start analysis
    poller = await document_intelligence_client.begin_analyze_document(
        "prebuilt-layout",
        analyze_request,
        output_content_format=DocumentContentFormat.MARKDOWN
        
    )
    
    result = await poller.result()
    page_wise_md = result.content.split("<!-- PageBreak -->")

    page_wise_ocr = []

    for page_idx, page in enumerate(result.pages):
        page_wise_context = ""
        if not(page.lines):
            import pdb;pdb.set_trace()
        for line_idx, line in enumerate(page.lines):
            page_wise_context += line.content + " "

        page_wise_ocr.append(page_wise_context)

    return page_wise_md, page_wise_ocr


In [5]:
page_wise_md, page_wise_ocr = await data_extractor(file_path)

# Text Cleaning Process:


## Handling the Header and Footer of the Documents:

In [6]:
# for i in range(len(page_wise_ocr)):
#     print(page_wise_ocr[i])

In [76]:
import re

def remove_header_footer_flexible(text):
    """
    More flexible version that can handle variations in the header/footer.
    Handles multiple header formats found in MBR documents.
    """
    
    # Flexible header patterns
    header_patterns = [
        # Main header pattern (page 1 style)
        r'Master Batch Record\s*30000773\s*[-–]\s*ARIPIPRAZOLE\s*5\s*MG\s*TAB\s*Effective\s*Alembic\s*Touching Lives over["\s]*100\s*years\s*ID/Version/Description\s*F1M00332/00000001/Aripiprazole Tab[\.\s]*USP 5mg',
        
        # Simpler catch-all for header
        r'Master Batch Record\s*30000773[^M]*?Aripiprazole Tab\.?USP 5mg',
        
        # NEW: Repeated header on subsequent pages (Material line)
        r'Master Batch Record\s*Material:\s*30000773\s*[-–]\s*ARIPIPRAZOLE\s*5\s*MG\s*TAB\s*BO',
        
        # NEW: Variation without "BO" at end
        r'Master Batch Record\s*Material:\s*30000773\s*[-–]\s*ARIPIPRAZOLE\s*5\s*MG\s*TAB(?=\s*BO|\s*$)',
        
        # NEW: Just the "Master Batch RecordMaterial:" line standalone
        r'Master Batch RecordMaterial:\s*30000773\s*[-–]\s*ARIPIPRAZOLE\s*5\s*MG\s*TABBO',
    ]
    
    # Flexible footer patterns
    footer_patterns = [
        # Standard footer: Page X of Y + Name + Date + Version
        r'Page\s*\d+\s*of\s*\d+\s*[A-Za-z\s]+\d{2}/\d{2}/\d{4}\s*\d{2}:\d{2}:\d{2}\s*V\d+',
        
        # Alternative: Version first
        r'V\d+\s*[A-Za-z\s]+\d{2}/\d{2}/\d{4}\s*\d{2}:\d{2}:\d{2}\s*Page\s*\d+\s*of\s*\d+',
        
        # Just page number pattern
        r'Page\s*\d+\s*of\s*\d+',

        # remove 
        r'group:---________________Verification signature:no',
        r'group:---Verification signature:no________________',
    ]
    
    cleaned_text = text
    
    # Remove all header patterns
    for pattern in header_patterns:
        cleaned_text = re.sub(pattern, ' ', cleaned_text, flags=re.IGNORECASE | re.DOTALL)
    
    # Remove all footer patterns
    for pattern in footer_patterns:
        cleaned_text = re.sub(pattern, ' ', cleaned_text, flags=re.IGNORECASE)
    
    # Clean up whitespace
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)
    cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text)
    cleaned_text = cleaned_text.strip(" ")
    
    return cleaned_text


In [77]:
cleaned_text_ocr = [
    remove_header_footer_flexible(text)
    for text in page_wise_ocr
]


In [78]:
len(cleaned_text_ocr)

198

In [79]:
cleaned_text = "\n\n".join(cleaned_text_ocr)

In [81]:
print(cleaned_text)

Standard batch size1,500,000 NOEstimated yield1,500,000 NOLong descriptionProduct Name : Aripiprazole Tablets USP 5 mgGeneric Name of Product : Aripiprazole Tablets USPBrand Name : NAStrength : 5 mgStandard Batch Size in Kg : 142.500 kgStandard Batch Size in Unit : 1,500,000 TabletsReference Document No. : MFC/0353-05Market/Customer : Export /USRef. BMR No. : F1\BMR\00837 & 3.0ApprovalStatusTextMaster Batch Record 10#F1M00332#Aripiprazole Tab.USP 5mg#30000773# --- ##00000001DraftObject was createdDate - User: 24/09/2025 15:57:34 - 27759 / Kinjal MehtaIn circulationPut MBR in circulationDate - User: 24/09/2025 16:30:55 - 27759 / Kinjal MehtaIn circulationMBR approval by technical support teamDate - User: 30/09/2025 12:26:34 - 16328 / Yesha PatelIn circulationMBR approval by production teamDate - User: 07/10/2025 14:08:16 - 9183 / Nilesh PatelReleasedMBR approval by QADate - User: 10/10/2025 13:20:40 - 11693 / Rahul JangaleEffectiveSet MBR effectiveDate - User: 16/10/2025 15:55:08 - 1558

In [82]:
## store in to txt files
with open(
    r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparsion-using-Azure-OCR\Backend\artifact\data.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(str(cleaned_text))

## Parent Chunking:

In [ ]:
## parent chunk statergy:
"""
-- Extract the data from the LLM, and OCR
-- Merge the data
-- ID - BO Description: {type}

"""

In [83]:
lst = ['INFO - Batch Information', 'DSPRM - Dispensing RM', 'GRAN - Granulation', 'COMP - Compression', 'INSCOM - Inspection']

In [84]:
empt_abbrevation_lst = []
for i in range(len(lst)):
    text = lst[i]
    text_list = text.split(" - ")
    empt_abbrevation_lst.append(text_list[0])
    
print(empt_abbrevation_lst)

['INFO', 'DSPRM', 'GRAN', 'COMP', 'INSCOM']


{'Approval_1': 'Standard batch size1,500,000 NOEstimated yield1,500,000 NOLong descriptionProduct Name : Aripiprazole Tablets USP 5 mgGeneric Name of Product : Aripiprazole Tablets USPBrand Name : NAStrength : 5 mgStandard Batch Size in Kg : 142.500 kgStandard Batch Size in Unit : 1,500,000 TabletsReference Document No. : MFC/0353-05Market/Customer : Export /USRef. BMR No. : F1\\BMR\\00837 & 3.0ApprovalStatusTextMaster Batch Record 10#F1M00332#Aripiprazole Tab.USP 5mg#30000773# --- ##00000001DraftObject was createdDate - User: 24/09/2025 15:57:34 - 27759 / Kinjal MehtaIn circulationPut MBR in circulationDate - User: 24/09/2025 16:30:55 - 27759 / Kinjal MehtaIn circulationMBR approval by technical support teamDate - User: 30/09/2025 12:26:34 - 16328 / Yesha PatelIn circulationMBR approval by production teamDate - User: 07/10/2025 14:08:16 - 9183 / Nilesh PatelReleasedMBR approval by QADate - User: 10/10/2025 13:20:40 - 11693 / Rahul JangaleEffectiveSet MBR effectiveDate - User: 16/10/20

In [ ]:
ALLOWED_PARENTS = [
    "INFO - Batch Information",
    "DSPRM - Dispensing RM",
    "GRAN - Granulation",
    "COMP - Compression",
    "INSCOM - Inspection"
]

## parent keywords need to extracted from the gpt4
PARENT_KEYWORD = "ID - BO Description"

In [85]:
def normalize_text(text: str) -> str:
    """Normalize OCR noise"""
    text = text.replace("\r", "\n")
    text = re.sub(r"\n+", "\n", text)
    return text.strip()


def detect_parent(line: str):
    """
    Detect which allowed parent this line belongs to
    """
    for parent in ALLOWED_PARENTS:
        if parent in line:
            return parent
    return None

In [99]:
from collections import defaultdict

def build_parent_chunks(raw_text: str):
    text = normalize_text(raw_text)
    # print(text)
    lines = text.split("\n")
    print(len(lines))
    
    parent_chunks = defaultdict(list)
    current_parent = "MISCELLANEOUS"

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Detect BO Description
        if PARENT_KEYWORD in line:
            detected = detect_parent(line)
            if detected:
                current_parent = detected
            else:
                current_parent = "MISCELLANEOUS"

        parent_chunks[current_parent].append(line)

    # -------------------------------
    # BUILD FINAL JSON
    # -------------------------------
    output = []
    parent_id = 1

    for parent_name, content_lines in parent_chunks.items():
        merged_text = "\n".join(content_lines).strip()

        output.append({
            "parent_chunk_id": parent_id,
            "parent_chunk_name": parent_name,
            "number_of_child_chunks_possible": len(content_lines),
            "chunk_data": merged_text
        })

        parent_id += 1

    return output


In [100]:
parent_chunks = build_parent_chunks(cleaned_text)

198


In [101]:
parent_chunks

[{'parent_chunk_id': 1,
  'parent_chunk_name': 'MISCELLANEOUS',
  'number_of_child_chunks_possible': 1,
  'chunk_data': 'Standard batch size1,500,000 NOEstimated yield1,500,000 NOLong descriptionProduct Name : Aripiprazole Tablets USP 5 mgGeneric Name of Product : Aripiprazole Tablets USPBrand Name : NAStrength : 5 mgStandard Batch Size in Kg : 142.500 kgStandard Batch Size in Unit : 1,500,000 TabletsReference Document No. : MFC/0353-05Market/Customer : Export /USRef. BMR No. : F1\\BMR\\00837 & 3.0ApprovalStatusTextMaster Batch Record 10#F1M00332#Aripiprazole Tab.USP 5mg#30000773# --- ##00000001DraftObject was createdDate - User: 24/09/2025 15:57:34 - 27759 / Kinjal MehtaIn circulationPut MBR in circulationDate - User: 24/09/2025 16:30:55 - 27759 / Kinjal MehtaIn circulationMBR approval by technical support teamDate - User: 30/09/2025 12:26:34 - 16328 / Yesha PatelIn circulationMBR approval by production teamDate - User: 07/10/2025 14:08:16 - 9183 / Nilesh PatelReleasedMBR approval by 

In [102]:
import json

with open(
    r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparsion-using-Azure-OCR\Backend\artifact\parent_chunking_sop.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(parent_chunks, f, indent=2, ensure_ascii=False)


## Child Chunking|

In [105]:
parent_chunks[2]

{'parent_chunk_id': 3,
 'parent_chunk_name': 'DSPRM - Dispensing RM',
 'number_of_child_chunks_possible': 29,
 'chunk_data': 'ID - BO Description: DSPRM - Dispensing RMActive: yes Reconciliation: noStart condition: Predecessors terminatedProduction unitIDDescriptionProduction areaDISPDispensingProduction PharmacyDetailed BOLevel Step!(1) F1M00332/DSPRM/INFO Batch Information Sheet Active: yes Optional: no (Common BF)Subtype:Date / signature for basic functionUser identification:No identificationUser ident. (2) F1M00332/DSPRM/INFO/PDETAIL Product Details Active: yes Optional: no (Common BF)Subtype:PRODD - Product Details Active: yes Subtype: None"For Reference"---1 Generic Name of Product : Aripiprazole Tablets USP2 Brand Name of Product : NA3 Label Claim : Each tablet contains 5 mg of Aripiprazole USP.4 Storage Condition: Store in tightly closed containers at 25℃ (77ºF); excursions permitted to 15°-30℃ (59°-86ºF) [see USP ControlledRoom Temperature].5 Stage /Dosage Form : Uncoated Tabl

In [106]:
data = parent_chunks[2].get("chunk_data")

In [107]:
print(data)

ID - BO Description: DSPRM - Dispensing RMActive: yes Reconciliation: noStart condition: Predecessors terminatedProduction unitIDDescriptionProduction areaDISPDispensingProduction PharmacyDetailed BOLevel Step!(1) F1M00332/DSPRM/INFO Batch Information Sheet Active: yes Optional: no (Common BF)Subtype:Date / signature for basic functionUser identification:No identificationUser ident. (2) F1M00332/DSPRM/INFO/PDETAIL Product Details Active: yes Optional: no (Common BF)Subtype:PRODD - Product Details Active: yes Subtype: None"For Reference"---1 Generic Name of Product : Aripiprazole Tablets USP2 Brand Name of Product : NA3 Label Claim : Each tablet contains 5 mg of Aripiprazole USP.4 Storage Condition: Store in tightly closed containers at 25℃ (77ºF); excursions permitted to 15°-30℃ (59°-86ºF) [see USP ControlledRoom Temperature].5 Stage /Dosage Form : Uncoated Tablet6 Market/Customer : Export /US7 Reference Document No. : MFC/0353-058 Manufacturing license No. : G/9599 Manufactured By : A

In [146]:
## split data arcorrding the 
id_pattern = r"(F1M00332+/DSPRM/[A-Z]+(?:/[A-Z]+)+(?:/[A-Z]+)?)"

In [147]:
matches = list(re.finditer(id_pattern, data))

In [148]:
matches

[<re.Match object; span=(390, 417), match='F1M00332/DSPRM/INFO/PDETAIL'>,
 <re.Match object; span=(1257, 1280), match='F1M00332/DSPRM/INFO/ABB'>,
 <re.Match object; span=(2918, 2944), match='F1M00332/DSPRM/INFO/SFINST'>,
 <re.Match object; span=(5668, 5694), match='F1M00332/DSPRM/INFO/GNINST'>,
 <re.Match object; span=(6426, 6449), match='F1M00332/DSPRM/INFO/MFC'>,
 <re.Match object; span=(10033, 10054), match='F1M00332/DSPRM/LC/LCD'>,
 <re.Match object; span=(10566, 10590), match='F1M00332/DSPRM/LC/INTACT'>,
 <re.Match object; span=(11294, 11316), match='F1M00332/DSPRM/LC/CAMP'>,
 <re.Match object; span=(12131, 12154), match='F1M00332/DSPRM/CAL/CALD'>,
 <re.Match object; span=(12674, 12696), match='F1M00332/DSPRM/CAL/CAL'>,
 <re.Match object; span=(12965, 12990), match='F1M00332/DSPRM/CAL/CAL/AR'>,
 <re.Match object; span=(15791, 15817), match='F1M00332/DSPRM/CAL/CAL/DAR'>,
 <re.Match object; span=(16476, 16501), match='F1M00332/DSPRM/CAL/CAL/AR'>,
 <re.Match object; span=(19351, 1937

In [149]:
data_info = parent_chunks[1].get("chunk_data")

In [150]:
print(data_info)

ID - BO Description: INFO - Batch InformationActive: yes Reconciliation: noStart condition: No restrictionProduction unitIDDescriptionProduction areaQADOCPRQA Document cellProduction PharmacyDetailed BOLevel Step!(1) F1M00332/INFO/BATCH Batch Information Active: yes Optional: no (Common BF)Subtype:1. Follow SOP No. F1\PRISOP\0274 for General instruction for MES. 2. Follow SOP No. F1\PRISOP\0275 for EBR execution in MES. 3.FollowSOP No. CIQAISOP\0018 for Good Documentation Practices.SRNO - Sr. No. of EBR Active: yes Subtype: None---Set valueActual valueEntry / SignatureCUST - Customer Batch No. Active: yes Subtype: None---1 Enter customer batch no. (if required) or Else write "NA".Set valueActual valueEntry / SignatureDate / signature for basic functionUser identification:No identificationUser ident. (1) F1M00332/INFO/CHD Change History Active: yes Optional: no (Common BF)Subtype:Change History of DocumentHISTORY - Change History Active: yes Subtype: None"For Reference"---1 Current BMR 

In [154]:
# id_pattern = r"(F1M00332+/INFO/[A-Z]+(?:/[A-Z]+)+(?:/[A-Z]+)?)"
# id_pattern = r"(F1M\d+/INFO/[A-Z]+(?:/[A-Z]+)?)"


id_pattern = r"(F1M\d+/(?:[A-Z0-9]+(?:/[A-Z0-9]+)*))"

# matches = list(re.finditer(id_pattern, data))

In [155]:
matches = list(re.finditer(id_pattern, data_info))

In [156]:
matches

[<re.Match object; span=(217, 236), match='F1M00332/INFO/BATCH'>,
 <re.Match object; span=(815, 832), match='F1M00332/INFO/CHD'>]

In [157]:
matches = list(re.finditer(id_pattern, data))

In [158]:
matches

[<re.Match object; span=(214, 233), match='F1M00332/DSPRM/INFO'>,
 <re.Match object; span=(390, 417), match='F1M00332/DSPRM/INFO/PDETAIL'>,
 <re.Match object; span=(1257, 1280), match='F1M00332/DSPRM/INFO/ABB'>,
 <re.Match object; span=(2918, 2944), match='F1M00332/DSPRM/INFO/SFINST'>,
 <re.Match object; span=(5668, 5694), match='F1M00332/DSPRM/INFO/GNINST'>,
 <re.Match object; span=(6426, 6449), match='F1M00332/DSPRM/INFO/MFC'>,
 <re.Match object; span=(9085, 9105), match='F1M00332/DSPRM/DATEC'>,
 <re.Match object; span=(9517, 9535), match='F1M00332/DSPRM/EVC'>,
 <re.Match object; span=(9868, 9885), match='F1M00332/DSPRM/LC'>,
 <re.Match object; span=(10033, 10054), match='F1M00332/DSPRM/LC/LCD'>,
 <re.Match object; span=(10566, 10590), match='F1M00332/DSPRM/LC/INTACT'>,
 <re.Match object; span=(11294, 11316), match='F1M00332/DSPRM/LC/CAMP'>,
 <re.Match object; span=(11956, 11974), match='F1M00332/DSPRM/CAL'>,
 <re.Match object; span=(12131, 12154), match='F1M00332/DSPRM/CAL/CALD'>,
 

In [163]:
dummy_matches = []
for i in range(len(parent_chunks)):
    id_pattern = r"(F1M\d+/(?!\d+$)[A-Z0-9]+(?:/[A-Z0-9]+)*)"

    matches = list(re.finditer(id_pattern, parent_chunks[i].get("chunk_data")))
    dummy_matches.append(matches)

In [164]:
dummy_matches

[[],
 [<re.Match object; span=(217, 236), match='F1M00332/INFO/BATCH'>,
  <re.Match object; span=(815, 832), match='F1M00332/INFO/CHD'>],
 [<re.Match object; span=(214, 233), match='F1M00332/DSPRM/INFO'>,
  <re.Match object; span=(390, 417), match='F1M00332/DSPRM/INFO/PDETAIL'>,
  <re.Match object; span=(1257, 1280), match='F1M00332/DSPRM/INFO/ABB'>,
  <re.Match object; span=(2918, 2944), match='F1M00332/DSPRM/INFO/SFINST'>,
  <re.Match object; span=(5668, 5694), match='F1M00332/DSPRM/INFO/GNINST'>,
  <re.Match object; span=(6426, 6449), match='F1M00332/DSPRM/INFO/MFC'>,
  <re.Match object; span=(9085, 9105), match='F1M00332/DSPRM/DATEC'>,
  <re.Match object; span=(9517, 9535), match='F1M00332/DSPRM/EVC'>,
  <re.Match object; span=(9868, 9885), match='F1M00332/DSPRM/LC'>,
  <re.Match object; span=(10033, 10054), match='F1M00332/DSPRM/LC/LCD'>,
  <re.Match object; span=(10566, 10590), match='F1M00332/DSPRM/LC/INTACT'>,
  <re.Match object; span=(11294, 11316), match='F1M00332/DSPRM/LC/CA

In [185]:
def split_chunks_clean(text):
    # Regex pattern for dynamic IDs
    id_pattern = re.compile(r"(F1M\d+/(?:[A-Z]+/)+[A-Z]+)")

    matches = list(id_pattern.finditer(text))

    chunks = []
    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)

        # Get the full text after ID
        full_text = text[start:end].strip()

        # Split name_chunk from chunk_data: take everything before the first capitalized word followed by a colon or known pattern
        name_match = re.match(r'([A-Za-z ]+?)\s(?=[A-Z][a-z]*:)', full_text)
        if name_match:
            name_chunk = name_match.group(1).strip()
            chunk_data = full_text[name_match.end():].strip()
        else:
            # fallback: take first words until "Active" appears
            split_pos = full_text.find("Active:")
            if split_pos != -1:
                name_chunk = full_text[:split_pos].strip()
                chunk_data = full_text[split_pos:].strip()
            else:
                name_chunk = full_text.split()[0]
                chunk_data = full_text[len(name_chunk):].strip()

        chunks.append({
            "child_chunk_id": match.group(),
            "name_chunk": name_chunk,
            "chunk_data": chunk_data
        })

    return chunks

In [186]:
split_chunks_clean(parent_chunks[1].get("chunk_data"))

[{'child_chunk_id': 'F1M00332/INFO/BATCH',
  'name_chunk': 'Batch Information',
  'chunk_data': 'Active: yes Optional: no (Common BF)Subtype:1. Follow SOP No. F1\\PRISOP\\0274 for General instruction for MES. 2. Follow SOP No. F1\\PRISOP\\0275 for EBR execution in MES. 3.FollowSOP No. CIQAISOP\\0018 for Good Documentation Practices.SRNO - Sr. No. of EBR Active: yes Subtype: None---Set valueActual valueEntry / SignatureCUST - Customer Batch No. Active: yes Subtype: None---1 Enter customer batch no. (if required) or Else write "NA".Set valueActual valueEntry / SignatureDate / signature for basic functionUser identification:No identificationUser ident. (1)'},
 {'child_chunk_id': 'F1M00332/INFO/CHD',
  'name_chunk': 'Change History',
  'chunk_data': 'Active: yes Optional: no (Common BF)Subtype:Change History of DocumentHISTORY - Change History Active: yes Subtype: None"For Reference"---1 Current BMR No. : F1\\BMR\\00837-1.0 Supersedes BMR No. : Nil Change Control No. : F1/PC2401662 Current

In [191]:
parent_chunks

[{'parent_chunk_id': 1,
  'parent_chunk_name': 'MISCELLANEOUS',
  'number_of_child_chunks_possible': 1,
  'chunk_data': 'Standard batch size1,500,000 NOEstimated yield1,500,000 NOLong descriptionProduct Name : Aripiprazole Tablets USP 5 mgGeneric Name of Product : Aripiprazole Tablets USPBrand Name : NAStrength : 5 mgStandard Batch Size in Kg : 142.500 kgStandard Batch Size in Unit : 1,500,000 TabletsReference Document No. : MFC/0353-05Market/Customer : Export /USRef. BMR No. : F1\\BMR\\00837 & 3.0ApprovalStatusTextMaster Batch Record 10#F1M00332#Aripiprazole Tab.USP 5mg#30000773# --- ##00000001DraftObject was createdDate - User: 24/09/2025 15:57:34 - 27759 / Kinjal MehtaIn circulationPut MBR in circulationDate - User: 24/09/2025 16:30:55 - 27759 / Kinjal MehtaIn circulationMBR approval by technical support teamDate - User: 30/09/2025 12:26:34 - 16328 / Yesha PatelIn circulationMBR approval by production teamDate - User: 07/10/2025 14:08:16 - 9183 / Nilesh PatelReleasedMBR approval by 

In [187]:
dummy_data_lst = []
for i in range(len(parent_chunks)):
    data = split_chunks_clean(parent_chunks[i].get("chunk_data"))
    dummy_data_lst.append(data)

In [190]:


with open(
    r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparsion-using-Azure-OCR\Backend\artifact\child_chunk.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(dummy_data_lst, f, indent=2, ensure_ascii=False)


In [192]:

import re

def split_child_chunks(text):
    """
    Splits a parent chunk's text into child chunks with:
    - child_chunk_id
    - name_of_child_chunk
    - child_chunk_data
    """

    id_pattern = re.compile(r"(F1M\d+/(?:[A-Z]+/)+[A-Z]+)")
    matches = list(id_pattern.finditer(text))

    child_chunks = []

    for i, match in enumerate(matches):
        chunk_id = match.group(1)

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        section_text = text[start:end].strip()

        # 🔑 Split name and data using " Active:"
        if " Active:" in section_text:
            name_part, data_part = section_text.split(" Active:", 1)
            name_of_child_chunk = name_part.strip()
            child_chunk_data = "Active:" + data_part.strip()
        else:
            # fallback (rare)
            name_of_child_chunk = section_text.split()[0]
            child_chunk_data = section_text[len(name_of_child_chunk):].strip()

        child_chunks.append({
            "chunk_id": chunk_id,
            "name_of_child_chunk": name_of_child_chunk,
            "child_chunk_data": child_chunk_data
        })

    return child_chunks


In [196]:
final_output = []

for parent in parent_chunks:
    parent_entry = {
        "name_of_parent_chunk": parent.get("parent_chunk_name"),
        "parent_chunk": parent.get("chunk_data"),
        "child_chunks": split_child_chunks(parent.get("chunk_data", ""))
    }
    final_output.append(parent_entry)


In [200]:
with open(
    r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparsion-using-Azure-OCR\Backend\artifact\child_chunk.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

In [ ]:
final_output

[{'name_of_parent_chunk': 'MISCELLANEOUS',
  'parent_chunk': 'Standard batch size1,500,000 NOEstimated yield1,500,000 NOLong descriptionProduct Name : Aripiprazole Tablets USP 5 mgGeneric Name of Product : Aripiprazole Tablets USPBrand Name : NAStrength : 5 mgStandard Batch Size in Kg : 142.500 kgStandard Batch Size in Unit : 1,500,000 TabletsReference Document No. : MFC/0353-05Market/Customer : Export /USRef. BMR No. : F1\\BMR\\00837 & 3.0ApprovalStatusTextMaster Batch Record 10#F1M00332#Aripiprazole Tab.USP 5mg#30000773# --- ##00000001DraftObject was createdDate - User: 24/09/2025 15:57:34 - 27759 / Kinjal MehtaIn circulationPut MBR in circulationDate - User: 24/09/2025 16:30:55 - 27759 / Kinjal MehtaIn circulationMBR approval by technical support teamDate - User: 30/09/2025 12:26:34 - 16328 / Yesha PatelIn circulationMBR approval by production teamDate - User: 07/10/2025 14:08:16 - 9183 / Nilesh PatelReleasedMBR approval by QADate - User: 10/10/2025 13:20:40 - 11693 / Rahul JangaleE

: 